# 06 - Deep Learning

Training a 1D CNN and LSTM for Human Activity Recognition using PyTorch.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

In [ ]:
X = np.load("../data/processed/X_windows.npy")
y = np.load("../data/processed/y_windows.npy")

# Adjust labels to start from 0
min_label = y.min()
y = y - min_label

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Create DataLoaders

In [ ]:
batch_size = 64

train_data = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_data = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, batch_size=batch_size)

## Define 1D CNN Model

In [ ]:
class ActivityCNN(nn.Module):
    def __init__(self, num_classes, num_features):
        super(ActivityCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.flatten = nn.Flatten()
        # Calculate output size dynamically or assume predefined window
        self.fc1 = nn.Linear(64 * 63, 128)  # for window=128
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # PyTorch expects input shape (batch, channels, seq_len)
        x = x.transpose(1, 2)  
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

num_classes = len(np.unique(y))
num_features = X.shape[2]
model = ActivityCNN(num_classes, num_features)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

## Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

## Evaluation

In [ ]:
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("CNN Accuracy:", accuracy_score(all_labels, all_preds))

## Save Model

In [ ]:
os.makedirs("../checkpoints/dl", exist_ok=True)
torch.save(model.state_dict(), "../checkpoints/dl/cnn_model.pth")
print("Deep Learning model saved.")